In [3]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "learnneo4j")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

QUERY = """
MATCH (me:Person {name: $name})-[:FRIEND]-(friend)-[:BOUGHT]->(book:Book)
WHERE NOT (me)-[:BOUGHT]->(book)
RETURN book.title AS title, count(*) AS score
ORDER BY score DESC
"""

records, summary, keys = driver.execute_query(QUERY, name="Linh")
for record in records:
    print(record["title"], record["score"])
print(f"{summary.result_available_after} ms")



Data Viz 1
2 ms


In [4]:
import neo4j

df = driver.execute_query(
    "MATCH (p:Person) RETURN p.name AS name, p.city AS city, p.age AS age",
    result_transformer_=neo4j.Result.to_df,
)
print(df.groupby("city")["age"].mean())

city
Can Tho    21.0
Da Nang    22.0
Hanoi      21.0
Hue        23.0
Name: age, dtype: float64


In [ ]:
people = [{"name": f"user{i}", "city": "Hanoi"} for i in range(10_000)]

driver.execute_query("""
    UNWIND $rows AS row
    MERGE (p:Person {name: row.name})
    SET p.city = row.city
""", rows=people)